# Sales Loss Prediction | 01. Synthetic data

## About the project
This project was born in my day-to-day work as a data scientist at a distributor of solar energy equipment. The customers are **solar installers** (in Portuguese, *integradores*): small and medium companies that buy photovoltaic kits (modules, inverters and structures) and install them for the end consumer.

Every quote becomes a sales order, and every order ends in one of two ways: **won** (*Ganho*) or **lost** (*Perda*). The commercial team has hundreds of open orders at the same time and no time to follow all of them closely, so the question asked was simple: **which open orders are most likely to be lost, and why?** With that answer the sales reps can prioritise their calls.

The model went to production with a daily scoring job, a weekly retraining job and an API consumed by the sales system. For this portfolio, the company data obviously cannot be published, so the whole project was rebuilt on **synthetic data** that follows the same structure (same tables, same columns, same business rules) as the original one.

This project is split into four notebooks:

| Notebook | Content |
|----------|---------|
| **01. Synthetic data** | How the fake database is generated |
| 02. Exploratory analysis | Descriptive analysis of the orders |
| 03. Features | Order-level and customer-history features, without leakage |
| 04. Model | XGBoost, evaluation, SHAP and scoring of the open orders |

## Why generate the data instead of using a public dataset?
I could have picked a public churn dataset, but the most interesting part of this project is not the algorithm, it is the **business structure** of the data: orders with several items, customers with history, prices that change over time, discounts that need approval. A generic dataset would lose all of that.

A side benefit of synthetic data is that **we know the truth**. Each order receives a "real" probability of being lost, which is saved separately. At the end of notebook 04 we can compare the model with the best result that would be theoretically possible, something that never happens with real data.

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 50505
rng = np.random.default_rng(SEED)

# Fixed reference date: it plays the role of "today" in the dataset.
# Using date.today() would make every run produce a different dataset.
DATA_INICIO = pd.Timestamp("2023-01-01")
DATA_REF = pd.Timestamp("2025-06-30")
N_INTEGRADORES = 1500

PASTA_DADOS = Path("../data")
PASTA_DADOS.mkdir(exist_ok=True)

## The installers (customers)
Each installer receives some characteristics. Part of them are **observable** (state, registration date) and part are **latent**, they exist only to generate the behaviour and will never be given to the model:

| Variable | Description |
|----------|-------------|
| `uf_cliente` | Brazilian state of the installer, weighted by the size of the solar market |
| `Data_Primeira_Compra` | Date of the first purchase (customer since) |
| `qualidade` | *Latent.* How "good" the customer is; good customers close more orders |
| `kit_habitual` | *Latent.* Usual kit size in kWp; small installers buy small kits |
| `pedidos_por_ano` | *Latent.* Purchase frequency |
| `pref_financiamento` | *Latent.* Probability of paying through financing |

In [2]:
ufs = ["SP", "MG", "PR", "RS", "SC", "GO", "BA", "RJ", "MT", "CE", "PE", "ES", "MS", "DF",
       "PA", "MA", "RN", "PB", "PI", "AL", "SE", "TO", "RO", "AM", "AC", "AP", "RR"]
peso_uf = np.array([16, 14, 8, 8, 6, 5, 5, 5, 4, 4, 3, 3, 3, 2,
                    2, 1.5, 1.5, 1.5, 1.2, 1, 1, 1, 0.8, 0.8, 0.3, 0.2, 0.2])
peso_uf = peso_uf / peso_uf.sum()

integradores = pd.DataFrame({
    "CpfCnpj": [f"INT{i:05d}" for i in range(1, N_INTEGRADORES + 1)],
    "uf_cliente": rng.choice(ufs, N_INTEGRADORES, p=peso_uf),
    # around 70% of the customers already existed when the data starts, the rest arrive over time
    "Data_Primeira_Compra": DATA_INICIO + pd.to_timedelta(rng.integers(-1500, 800, N_INTEGRADORES), unit="D"),
    "qualidade": rng.normal(0, 1, N_INTEGRADORES),
    "kit_habitual": rng.lognormal(np.log(8), 0.6, N_INTEGRADORES),
    "pedidos_por_ano": rng.gamma(2, 3, N_INTEGRADORES),
    "pref_financiamento": rng.beta(2, 5, N_INTEGRADORES),
})
integradores.head()

,CpfCnpj,uf_cliente,Data_Primeira_Compra,qualidade,kit_habitual,pedidos_por_ano,pref_financiamento
0,INT00001,DF,2019-05-20,-0.853030,21.242971,4.535374,0.345533
1,INT00002,SC,2019-05-28,0.650251,16.485242,7.788503,0.379219
2,INT00003,SP,2024-02-10,0.840326,2.972190,5.471448,0.213993
3,INT00004,PR,2020-09-22,-0.119378,2.020141,5.803272,0.553555
4,INT00005,SC,2024-04-17,1.341246,32.207791,2.832477,0.199144


## The orders
The number of orders of each installer follows a Poisson distribution, proportional to its purchase frequency and to the time it is active in the period. The creation date is uniform inside that active window.

In [3]:
inicio_ativo = integradores["Data_Primeira_Compra"].clip(lower=DATA_INICIO)
anos_ativos = (DATA_REF - inicio_ativo).dt.days / 365.25
n_pedidos = rng.poisson(integradores["pedidos_por_ano"] * anos_ativos)

pedidos = integradores.loc[integradores.index.repeat(n_pedidos)].reset_index(drop=True)
inicio = inicio_ativo.repeat(n_pedidos).reset_index(drop=True)
dias_janela = (DATA_REF - inicio).dt.days
pedidos["DataCriacaoId"] = inicio + pd.to_timedelta((rng.random(len(pedidos)) * dias_janela).astype(int), unit="D")

pedidos = pedidos.sort_values("DataCriacaoId", kind="mergesort").reset_index(drop=True)
pedidos["PedidoId"] = np.arange(100001, 100001 + len(pedidos))
pedidos["NumeroPedido"] = pedidos["PedidoId"] + 400000

print(f"{len(pedidos):,} orders from {pedidos['CpfCnpj'].nunique():,} installers")

19,066 orders from 1,456 installers


### Kit size and price
The kit size varies around the installer's usual size. The price per watt (R$/W) follows a **market price** that falls over time (as it really happened with solar equipment in Brazil), has a small seasonality and is higher in the North and Northeast because of freight. Each order has its own markup over that market price.

In [4]:
pedidos["potencia_kit_alvo"] = (pedidos["kit_habitual"] * rng.lognormal(0, 0.35, len(pedidos))).clip(1.5, 300)

anos = (pedidos["DataCriacaoId"] - DATA_INICIO).dt.days / 365.25
mes = pedidos["DataCriacaoId"].dt.month
fator_regiao = np.select(
    [pedidos["uf_cliente"].isin(["PA", "MA", "RN", "PB", "PI", "AL", "SE", "TO", "RO", "AM", "AC", "AP", "RR", "BA", "CE", "PE"]),
     pedidos["uf_cliente"].isin(["PR", "RS", "SC"])],
    [1.06, 0.97],
    default=1.0,
)
pedidos["rpw_mercado"] = (3.1 - 0.35 * anos + 0.06 * np.sin(2 * np.pi * mes / 12)) * fator_regiao
pedidos["markup"] = rng.normal(0, 0.07, len(pedidos))
pedidos["rpw_pedido"] = pedidos["rpw_mercado"] * (1 + pedidos["markup"])

### Discount, payment and freight
- **Discount:** about half of the orders have no discount, 40% have a discount between 1% and 8%, and 5% have a surcharge (*acréscimo*). Discounts above 4% (and some smaller ones) go to an approval analysis, and the chance of being rejected grows with the size of the discount.
- **Payment:** the installer's financing preference defines whether the order is financed; the remaining orders are split between cash, credit card and other forms. The labels imitate the free-text field of the real system, where the same thing is written in many ways.
- **Freight:** CIF (paid by the seller) or FOB (paid by the buyer).

In [5]:
n = len(pedidos)
tipo = rng.choice(["sem", "desconto", "acrescimo"], n, p=[0.55, 0.40, 0.05])
pedidos["perc_desconto_alvo"] = np.select(
    [tipo == "desconto", tipo == "acrescimo"],
    [-rng.uniform(0.01, 0.08, n), rng.uniform(0.01, 0.03, n)],
    default=0.0,
)

vai_para_analise = (pedidos["perc_desconto_alvo"] < -0.04) | ((pedidos["perc_desconto_alvo"] < 0) & (rng.random(n) < 0.2))
prob_reprovar = 1 / (1 + np.exp(-(-pedidos["perc_desconto_alvo"] - 0.065) * 70))
reprovado = rng.random(n) < prob_reprovar
pedidos["situacao_pedido"] = np.where(vai_para_analise, np.where(reprovado, "Reprovado", "Aprovado"), None)

formas = {
    "avista": ["PIX", "BOLETO A VISTA", "TED", "BOLETO + PIX"],
    "financiamento": ["FINANCIAMENTO BANCARIO", "FINANCIAMENTO + BOLETO", "FINANCIAMENTO INTERNO"],
    "cartao": ["CARTAO DE CREDITO", "PIX + CARTAO DE CREDITO"],
    "outros": ["PERMUTA", "A DEFINIR"],
}
financia = rng.random(n) < pedidos["pref_financiamento"]
grupo_pag = np.where(financia, "financiamento", rng.choice(["avista", "cartao", "outros"], n, p=[0.75, 0.18, 0.07]))
pedidos["grupo_pagamento"] = grupo_pag
pedidos["forma_pagamento"] = [rng.choice(formas[g]) for g in grupo_pag]

pedidos["TipoFrete"] = rng.choice(["CIF", "FOB"], n, p=[0.7, 0.3])

## The outcome: won or lost
This is the heart of the generator. The probability of loss is a logistic function of the order and customer characteristics. The coefficients were chosen so that the effects make business sense:

| Effect | Direction |
|--------|-----------|
| Markup over the market price | Expensive orders are lost more |
| Discount rejected | Strongly increases the loss |
| Discount approved | Slightly reduces the loss |
| Financing / card / other payment | Financing depends on a bank approval, so it is lost more |
| Customer quality (latent) | Good customers lose less |
| Kit far from the usual size | Unusual orders are lost more |
| FOB freight | Slightly increases the loss |
| December and January | Holiday season, more losses |
| New customer (< 90 days) | Increases the loss |

$$P(\text{loss}) = \frac{1}{1 + e^{-z}}$$

The value of $z$ for each order is saved as `prob_real`, the "truth" that the model will try to recover.

In [6]:
tempo_cliente = (pedidos["DataCriacaoId"] - pedidos["Data_Primeira_Compra"]).dt.days

z = (
    -1.15
    + 6.0 * pedidos["markup"]
    + 1.4 * (pedidos["situacao_pedido"] == "Reprovado")
    - 0.3 * (pedidos["situacao_pedido"] == "Aprovado")
    + np.select([grupo_pag == "financiamento", grupo_pag == "cartao", grupo_pag == "outros"], [0.6, 0.3, 0.5], default=0.0)
    - 0.9 * pedidos["qualidade"]
    + 0.5 * np.abs(np.log(pedidos["potencia_kit_alvo"] / pedidos["kit_habitual"]))
    + 0.25 * (pedidos["TipoFrete"] == "FOB")
    + 0.3 * mes.isin([12, 1])
    + 0.25 * (tempo_cliente < 90)
)
pedidos["prob_real"] = 1 / (1 + np.exp(-z))
perda = rng.random(n) < pedidos["prob_real"]

print(f"Average real probability of loss: {pedidos['prob_real'].mean():.1%}")

Average real probability of loss: 39.3%


### Time until the outcome
Won orders close faster than lost ones: when the customer wants the kit, payment comes quickly; when the customer gives up, the order is usually forgotten for some weeks until the sales rep marks it as lost. Financed orders take longer because of the bank analysis.

If the outcome date falls after the reference date, the order is still **open** (*Aberto*) and its outcome is unknown. These open orders are the ones that the model will score at the end.

This difference in closing time is **real** (it existed in the original data) and it will be very important in notebook 03, because it creates a subtle leakage trap.

In [7]:
tempo_ganho = 1 + rng.gamma(2, 5, n) + 8 * financia
tempo_perda = 1 + rng.gamma(2, 12, n)
tempo_ate_desfecho = np.where(perda, tempo_perda, tempo_ganho).clip(max=180).astype(int)

data_desfecho = pedidos["DataCriacaoId"] + pd.to_timedelta(tempo_ate_desfecho, unit="D")
aberto = data_desfecho > DATA_REF

pedidos["Status_Pedido"] = np.select([aberto, perda], ["Aberto", "Perda"], default="Ganho")
pedidos["DataGanhoPerdaID"] = data_desfecho.where(~aberto)

pedidos["Status_Pedido"].value_counts()

Status_Pedido
Ganho     11412
Perda      7270
Aberto      384
Name: count, dtype: int64

### Order stage and loss reason
The system keeps the **current stage** of each order (*descricao_etapa*). Notice that for closed orders the stage is a picture taken **after** the outcome: won orders are almost always at the last stage, while lost orders stay frozen at the stage where they died. This column must never be a model feature, and notebook 04 shows what happens if it is.

In [8]:
etapas_abertas = ["3/9 - Proposta enviada", "4/9 - Negociação de condições", "5/9 - Aguardando pagamento", "6/9 - Análise de crédito"]

pedidos["descricao_etapa"] = np.select(
    [pedidos["Status_Pedido"] == "Ganho", pedidos["Status_Pedido"] == "Perda"],
    [np.where(rng.random(n) < 0.9, "7/9 - Pagamento aprovado", "6/9 - Análise de crédito"),
     rng.choice(etapas_abertas[:3], n, p=[0.5, 0.3, 0.2])],
    default=rng.choice(etapas_abertas, n),
)

motivo_sorteado = rng.choice(
    ["Preço", "Desistência do cliente", "Comprou com concorrente", "Prazo de entrega", "Crédito não aprovado"],
    n, p=[0.3, 0.25, 0.2, 0.15, 0.1],
)
motivo = np.select(
    [pedidos["situacao_pedido"] == "Reprovado", (grupo_pag == "financiamento") & (rng.random(n) < 0.4), pedidos["markup"] > 0.08],
    ["Desconto reprovado", "Crédito não aprovado", "Preço"],
    default=motivo_sorteado,
)
# a few administrative losses (duplicated or test orders), they exist in any real CRM
motivo = np.where(rng.random(n) < 0.015, rng.choice(["Pedido duplicado", "Teste"], n, p=[0.7, 0.3]), motivo)
pedidos["motivo_perda"] = np.where(pedidos["Status_Pedido"] == "Perda", motivo, None)

## The items
The real table is at **item level**: one row per product of the order. Each kit has three lines:

- **Modules:** `PotenciakWp` below 1 (a module has 0.55 to 0.67 kWp), quantity enough to reach the kit size.
- **Inverter:** `PotenciakWp` above 1, sized with an *oversize* (module power / inverter power) between 1.05 and 1.45, as installers usually do.
- **Structure and cables:** `PotenciakWp` equal to 0.

This rule (module < 1 kWp < inverter) is exactly how the original pipeline separates the products, so it was kept.

In [9]:
pot_modulo = rng.choice([0.55, 0.58, 0.61, 0.665], n)
qtd_modulo = np.ceil(pedidos["potencia_kit_alvo"] / pot_modulo).astype(int)
potencia_kit = qtd_modulo * pot_modulo

tamanhos_inv = np.array([3, 5, 6, 8, 10, 12, 15, 20, 25, 30, 50, 75])
inv_necessario = potencia_kit / rng.uniform(1.05, 1.45, n)
qtd_inv = np.ceil(inv_necessario / 75).astype(int)
por_inversor = inv_necessario / qtd_inv
pot_inv = tamanhos_inv[np.minimum(np.searchsorted(tamanhos_inv, por_inversor * 0.95), len(tamanhos_inv) - 1)]

subtotal_pedido = potencia_kit * 1000 * pedidos["rpw_pedido"]
desconto_pedido = subtotal_pedido * pedidos["perc_desconto_alvo"]

colunas_pedido = ["PedidoId", "NumeroPedido", "CpfCnpj", "uf_cliente", "Data_Primeira_Compra", "DataCriacaoId",
                  "DataGanhoPerdaID", "Status_Pedido", "descricao_etapa", "motivo_perda", "forma_pagamento",
                  "TipoFrete", "situacao_pedido"]

itens_modulo = pedidos[colunas_pedido].assign(
    DescrProd=[f"Módulo fotovoltaico {int(p * 1000)}W" for p in pot_modulo],
    PotenciakWp=pot_modulo, Quantidade=qtd_modulo, SubTotal=subtotal_pedido * 0.60)
itens_inversor = pedidos[colunas_pedido].assign(
    DescrProd=[f"Inversor {p}kW" for p in pot_inv],
    PotenciakWp=pot_inv.astype(float), Quantidade=qtd_inv, SubTotal=subtotal_pedido * 0.30)
itens_estrutura = pedidos[colunas_pedido].assign(
    DescrProd="Estrutura e cabos",
    PotenciakWp=0.0, Quantidade=1, SubTotal=subtotal_pedido * 0.10)

itens = pd.concat([itens_modulo, itens_inversor, itens_estrutura], ignore_index=True)
# the discount is spread over the items in proportion to their value
itens["Desconto"] = itens["SubTotal"] * np.tile(pedidos["perc_desconto_alvo"].to_numpy(), 3)
itens["ValorTotal"] = itens["SubTotal"] + itens["Desconto"]
itens[["SubTotal", "Desconto", "ValorTotal"]] = itens[["SubTotal", "Desconto", "ValorTotal"]].round(2)
itens = itens.sort_values(["PedidoId", "PotenciakWp"], ascending=[True, False], kind="mergesort").reset_index(drop=True)

itens.head(6)

,PedidoId,NumeroPedido,CpfCnpj,uf_cliente,Data_Primeira_Compra,DataCriacaoId,DataGanhoPerdaID,Status_Pedido,descricao_etapa,motivo_perda,forma_pagamento,TipoFrete,situacao_pedido,DescrProd,PotenciakWp,Quantidade,SubTotal,Desconto,ValorTotal
0,100001,500001,INT00100,MA,2020-10-05,2023-01-01,2023-01-11,Ganho,7/9 - Pagamento aprovado,NaN,FINANCIAMENTO BANCARIO,FOB,NaN,Inversor 8kW,8.00,1,10073.57,0.0,10073.57
1,100001,500001,INT00100,MA,2020-10-05,2023-01-01,2023-01-11,Ganho,7/9 - Pagamento aprovado,NaN,FINANCIAMENTO BANCARIO,FOB,NaN,Módulo fotovoltaico 550W,0.55,18,20147.13,0.0,20147.13
2,100001,500001,INT00100,MA,2020-10-05,2023-01-01,2023-01-11,Ganho,7/9 - Pagamento aprovado,NaN,FINANCIAMENTO BANCARIO,FOB,NaN,Estrutura e cabos,0.00,1,3357.86,0.0,3357.86
3,100002,500002,INT00111,SP,2021-05-23,2023-01-01,2023-01-12,Perda,3/9 - Proposta enviada,Comprou com concorrente,BOLETO + PIX,FOB,NaN,Inversor 5kW,5.00,1,4149.74,0.0,4149.74
4,100002,500002,INT00111,SP,2021-05-23,2023-01-01,2023-01-12,Perda,3/9 - Proposta enviada,Comprou com concorrente,BOLETO + PIX,FOB,NaN,Módulo fotovoltaico 610W,0.61,7,8299.49,0.0,8299.49
5,100002,500002,INT00111,SP,2021-05-23,2023-01-01,2023-01-12,Perda,3/9 - Proposta enviada,Comprou com concorrente,BOLETO + PIX,FOB,NaN,Estrutura e cabos,0.00,1,1383.25,0.0,1383.25


## Saving
Two files are saved:

- `pedidos_itens.parquet`: the item table, exactly what the model pipeline would receive from the database.
- `verdade.parquet`: the real probability of loss of each order. It is **never** used for training, only for the final comparison in notebook 04.

In [10]:
itens.to_parquet(PASTA_DADOS / "pedidos_itens.parquet", index=False)
pedidos[["PedidoId", "prob_real"]].to_parquet(PASTA_DADOS / "verdade.parquet", index=False)

resumo = pd.DataFrame({
    "Item": ["Installers", "Orders", "Item rows", "Won", "Lost", "Open", "Period"],
    "Value": [f"{itens['CpfCnpj'].nunique():,}", f"{itens['PedidoId'].nunique():,}", f"{len(itens):,}",
              f"{(pedidos['Status_Pedido'] == 'Ganho').sum():,}", f"{(pedidos['Status_Pedido'] == 'Perda').sum():,}",
              f"{(pedidos['Status_Pedido'] == 'Aberto').sum():,}",
              f"{pedidos['DataCriacaoId'].min():%Y-%m-%d} to {pedidos['DataCriacaoId'].max():%Y-%m-%d}"],
})
resumo

,Item,Value
0,Installers,"1,456"
1,Orders,"19,066"
2,Item rows,"57,198"
3,Won,"11,412"
4,Lost,"7,270"
5,Open,384
6,Period,2023-01-01 to 2025-06-29


In [11]:
#Cleaning the environment
import gc
del integradores, pedidos, itens, itens_modulo, itens_inversor, itens_estrutura
gc.collect();